# Dorado HTTP 训练入口：阅读版

本轮资源尝试已暂停。这份 Notebook 说明旧两卡基线的控制流程，没有执行任何单元格，也不会申请 GPU。公开服务与路径是匿名占位值。

CPU 持久化，GPU 只读镜像／模型目录和本地盘。运行代码之前需要明确正确资源入口、实际实例、网络权限和完整离线包。凭据由环境注入，不能写入 Notebook。

## 第一步：资产与端点身份

下面保留原代码，检查解包清单、读取 run ID 和受控凭据，再发现两个逻辑槽位。端点不存在表示尚不能调用训练；端点数不会扩大 GPU 配额。

In [ ]:
import json, os, sys
from pathlib import Path
package = Path(os.environ['LOWBAND_PACKAGE_DIR']).resolve()
assert (package / 'package-manifest.json').is_file(), 'Expected the unpacked offline package'
sys.path.insert(0, str(package))
run_id = os.environ['LOWBAND_RUN_ID']
token = os.environ['LOWBAND_HTTP_TOKEN']
assert len(token) >= 24
from lowband_training.dorado_discovery import BernardEndpoints
from lowband_training.http_probe import probe
services = {
    'psm': '<configured-psm>',
    'services': {'hl':'<configured-service>', 'lf':'<configured-service>'},
    'region': 'cn', 'cluster': ['default'], 'env_type': ['prod'], 'refresh_seconds': 30,
}
provider = BernardEndpoints(**services)
endpoints = provider()
print({'discovered_endpoint_counts': provider.counts, 'selected_slots': sorted(endpoints)})
assert set(endpoints) == {'hl', 'lf'}, 'Wait for both task-3 services to receive resources'


## 第二步：分别验证 HTTP 与持久存储

下面代码对每个槽位做回环与身份检查。它只验证数据通路，不证明 CUDA 训练、模型质量或副本释放。

In [ ]:
probe_reports = {}
for slot, endpoint in endpoints.items():
    result = probe(endpoint, token,
        '<controller-storage-uri><operator>/lowband_training/http-validation')
    assert result['http_verified'] and result['hdfs_verified'] and result['worker'] == slot
    probe_reports[slot] = result
print(json.dumps(probe_reports, indent=2))


## 第三步：唯一控制器运行与恢复

只允许一份控制循环。从相同 run／配置恢复；收齐 delta、Adam、RNG 和游标后才发布全局版本。不能在两个 Notebook 或 Spark 的可重试 task 中并发运行。

In [ ]:
import torch
from lowband_training.hdfs_store import HdfsStore
from lowband_training.http_controller import Controller
assert set(probe_reports) == {'hl', 'lf'}
torch.set_num_threads(2)
config = json.loads((package / 'config.json').read_text())
store = HdfsStore(
    '<controller-storage-uri><operator>/lowband_training/runs/' + run_id,
    '/tmp/lowband-controller-' + run_id,
)
controller = Controller(store, config, provider, token, package / 'assets', run_id)
completed = controller.run()
print(json.dumps(completed, indent=2))


## 输出该怎样理解

completed.json 只说明状态提交，之后还需完整验证、恢复比较、接受 token 和资源释放证据。此 Notebook 在 Dorado Executor 未完成实测，不把开发机控制端的成功写成 Dorado 成功。